# Employee Attrition Prediction Using Random Forest

## Project Overview
This project uses the IBM HR Analytics Employee Attrition dataset to predict whether an employee is likely to leave the company (`Attrition`: Yes/No) using a Random Forest Classifier.

**Problem type:** Binary Classification  
**Dataset:** IBM HR Analytics Employee Attrition & Performance  
**Algorithm:** Random Forest Classifier

### Objectives
- Explore and understand the dataset.
- Clean and prepare the data.
- Encode categorical variables and split the data into training and testing sets.
- Train a Random Forest model.
- Evaluate the model using classification metrics.
- Interpret feature importance and summarize the findings.

**Dataset source:** [Kaggle – IBM HR Analytics Employee Attrition & Performance](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)

> Place `WA_Fn-UseC_-HR-Employee-Attrition.csv` in the same folder as this notebook before running it.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay
)

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

## 2. Load the Dataset

In [ ]:
# Read the CSV file from the same directory as this notebook
file_path = "WA_Fn-UseC_-HR-Employee-Attrition.csv"
df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

## 3. Basic Data Exploration

In [ ]:
# Dataset dimensions, column names, data types, and summary statistics
print("Number of rows and columns:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())

print("\\nData types and non-null counts:")
df.info()

print("\\nNumerical summary:")
display(df.describe().T)

print("\\nCategorical summary:")
display(df.describe(include="object").T)

In [ ]:
# Check target class distribution
print("Attrition distribution:")
display(df["Attrition"].value_counts())

print("\\nAttrition percentage:")
display((df["Attrition"].value_counts(normalize=True) * 100).round(2))

df["Attrition"].value_counts().plot(kind="bar", rot=0)
plt.title("Employee Attrition Distribution")
plt.xlabel("Attrition")
plt.ylabel("Number of Employees")
plt.show()

## 4. Data Quality Checks and Preprocessing

In [ ]:
# Check missing values and duplicate rows
print("Missing values in each column:")
display(df.isnull().sum().sort_values(ascending=False))

print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
# Remove duplicate rows if present
df = df.drop_duplicates().copy()

# Inspect columns with only one unique value
constant_columns = [col for col in df.columns if df[col].nunique(dropna=False) <= 1]
print("Constant columns:", constant_columns)

# Remove constant columns because they do not provide predictive information
df = df.drop(columns=constant_columns, errors="ignore")

# EmployeeNumber is an identifier, not an employee attribute useful for general prediction
df = df.drop(columns=["EmployeeNumber"], errors="ignore")

# Handle missing values:
# - Numeric columns: replace missing values with median
# - Categorical columns: replace missing values with most frequent value
numeric_columns = df.select_dtypes(include=["number"]).columns
categorical_columns = df.select_dtypes(include=["object", "category"]).columns

for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Shape after preprocessing:", df.shape)
print("Remaining missing values:", int(df.isnull().sum().sum()))

## 5. Split Features and Target

The target variable is `Attrition`. It contains two classes:
- `No`: Employee stayed.
- `Yes`: Employee left.

The identifier and constant columns were removed in the previous step. Categorical features will be one-hot encoded within a pipeline to keep preprocessing consistent and avoid data leakage.

In [ ]:
X = df.drop(columns=["Attrition"])
y = df["Attrition"].map({"No": 0, "Yes": 1})

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Target class counts:")
display(y.value_counts().rename(index={0: "No", 1: "Yes"}))

## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## 7. Encode Categorical Features and Build the Random Forest Model

In [ ]:
# Identify categorical and numeric feature columns
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()

# One-hot encode categorical features; numeric features pass through unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features)
    ]
)

# Random Forest classifier
random_forest = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1
)

# Combine preprocessing and model in a single pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", random_forest)
])

model

## 8. Train the Model

In [ ]:
model.fit(X_train, y_train)
print("Random Forest model trained successfully.")

## 9. Make Predictions

In [ ]:
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

print("First 10 actual labels:", y_test.iloc[:10].tolist())
print("First 10 predictions:", y_pred[:10].tolist())

## 10. Evaluate the Model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_probability)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, roc_auc]
})

display(metrics.round(4))
print("\\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Attrition", "Attrition"], zero_division=0))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Attrition", "Attrition"])
disp.plot()
plt.title("Random Forest - Confusion Matrix")
plt.show()

### ROC Curve

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_probability)
plt.title("Random Forest - ROC Curve")
plt.show()

## 11. Feature Importance

Feature importance indicates how much each encoded feature contributes to the trained Random Forest model. These scores describe model behavior and should not be interpreted as proof of cause and effect.

In [ ]:
# Retrieve the feature names after one-hot encoding
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
importances = model.named_steps["classifier"].feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

display(feature_importance.head(15))

# Plot the 15 most important features
top_features = feature_importance.head(15).sort_values("Importance")
plt.figure(figsize=(10, 7))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Top 15 Feature Importances - Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 12. Conclusion

In [ ]:
print("PROJECT CONCLUSION")
print("-----------------")
print(f"The Random Forest classifier was trained to predict employee attrition.")
print(f"Test accuracy:  {accuracy:.3f}")
print(f"Test precision: {precision:.3f}")
print(f"Test recall:    {recall:.3f}")
print(f"Test F1-score:  {f1:.3f}")
print(f"Test ROC-AUC:   {roc_auc:.3f}")

print("\\nThe feature-importance table above shows which input variables contributed most")
print("to the model's predictions. Model performance should be interpreted alongside")
print("the class distribution and the confusion matrix, particularly because attrition")
print("may be less common than non-attrition.")

## 13. Project Summary

- **Dataset:** IBM HR Analytics Employee Attrition & Performance
- **Problem:** Binary classification of employee attrition
- **Target:** `Attrition`
- **Preprocessing:** Duplicate and constant-column checks, missing-value handling, one-hot encoding
- **Model:** Random Forest Classifier
- **Evaluation:** Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and ROC-AUC
- **Interpretation:** Feature importance visualization

**Dataset source:** [Kaggle – IBM HR Analytics Employee Attrition & Performance](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)